# Лабораторная работа 3. Градиентный спуск, обусловленность и регуляризация

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 2 |
| Опора на лекции | лекция 2: градиентный спуск (опр. 2.1) и условие сходимости $0<\eta<1/\lambda_{\max}$ (утв. 2.2), SGD (опр. 2.4), мультиколлинеарность и неустойчивость решения (утв. 2.5), гребневая регрессия (опр. 2.7, теорема 2.9), MAP (опр. 2.10, утв. 2.11), LASSO (опр. 2.12); лекция 1: нормальные уравнения, матричное дифференцирование |
| Трудоёмкость | 2 ч аудиторно (части 1–3) + 6 ч самостоятельно |

## Цель работы

Реализовать градиентный спуск и его стохастический вариант, проверить границу шага $1/\lambda_{\max}$ из утверждения 2.2 и связь скорости сходимости с числом обусловленности; увидеть, как мультиколлинеарность разрушает решение МНК, и как обе регуляризации это лечат; реализовать LASSO покоординатным спуском и объяснить, откуда берётся разреженность.

## Что нужно сдать

Заполненный ноутбук `lab03_student.ipynb`, в котором:

1. выполнены все задания (ячейки с `# TODO`), код запускается сверху вниз без ошибок;
2. под каждым заданием заполнены ячейки **Вывод** — своими словами, не пересказ кода;
3. в конце — раздел «Итоги работы» с ответами на контрольные вопросы;
4. все графики подписаны (заголовок, оси, легенда).

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже). Отчёт с чужим вариантом не принимается.

> **О нормировке.** В §1 лекции 2 функционал берётся без усреднения:
> $Q(\theta) = \|X\theta - y\|^2$, поэтому $\nabla Q = 2X^{\mathsf T}(X\theta - y)$
> и граница шага равна $1/\lambda_{\max}$. В определении 2.4 (SGD) используется
> усреднённая форма $Q = \frac1\ell\sum_i Q_i$. Эти два $Q$ различаются множителем
> $\ell$, и потому «правильный» шаг для них отличается в $\ell$ раз.
> **Всегда проверяйте, какая нормировка у вашего $Q$** — это самая частая причина
> «почему у меня градиентный спуск разошёлся».
> Ниже нормировка указана явно в каждой части.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from scipy import optimize
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=3)
describe_variant(variant)

---
# Часть 1. Градиентный спуск и граница шага

Определение 2.1: $\theta^{(t+1)} = \theta^{(t)} - \eta_t \nabla Q(\theta^{(t)})$.
Для $Q(\theta) = \|X\theta - y\|^2$ имеем $\nabla Q(\theta) = 2X^{\mathsf T}(X\theta - y)$.

Утверждение 2.2 обещает сходимость при $0 < \eta < 1/\lambda_{\max}$, где
$\lambda_{\max}$ — наибольшее собственное число $X^{\mathsf T}X$. Проверим границу
экспериментально: возьмём шаги $\eta = c/\lambda_{\max}$ при
$c \in \{0.1,\ 0.5,\ 0.9,\ 1.0,\ 1.05\}$.

Идея доказательства подсказывает, что именно смотреть: ошибка в собственном
базисе умножается на $|1 - 2\eta\lambda_i|$, поэтому при $\eta = 1/\lambda_{\max}$
множитель равен ровно $1$ (колебания без затухания), а при большем $\eta$ —
превышает $1$ (расходимость).

In [ ]:
def grad_Q(theta, X, y):
    """Градиент Q(theta) = ||X theta - y||^2."""
    raise NotImplementedError


def gradient_descent(X, y, eta, n_iter=300, theta0=None):
    """Возвращает траекторию (n_iter + 1, p) метода из определения 2.1."""
    raise NotImplementedError


n = 200
x1 = rng.normal(0, 1, n)
x2 = 0.6 * x1 + 0.8 * rng.normal(0, 1, n)
X_gd = np.column_stack([x1, x2])
theta_true = np.array([2.0, -1.0])
y_gd = X_gd @ theta_true + rng.normal(0, 0.5, n)

# TODO: 1) найдите theta* через lstsq и lambda_max = max собственное число X^T X;
#       2) для c in [0.1, 0.5, 0.9, 1.0, 1.05] запустите ГС с шагом c / lambda_max;
#       3) на одном графике (полулогарифмическом по оси Q) покажите
#          Q(theta^(t)) - Q(theta*) для каждого шага.
#       Осторожно: при расходимости значения станут inf -- замаскируйте их np.nan.

### Задание 1.2. Траектории на линиях уровня

Постройте линии уровня $Q(\theta)$ на плоскости $(\theta_1, \theta_2)$ и нанесите
траектории спуска для двух шагов: маленького и близкого к границе. Отметьте
$\theta^*$.

In [ ]:
# TODO: линии уровня Q(theta) на сетке вокруг theta* и две траектории спуска
#       (маленький шаг и шаг у границы). Отметьте theta*.

> **Вывод.** При каком $c$ спуск перестал сходиться? Совпало ли это с границей утверждения 2.2? Как выглядит траектория при шаге, близком к границе, и почему она такая?
>
> *(ваш ответ здесь)*

---
# Часть 2. Скорость сходимости и число обусловленности

Из того же разложения по собственным векторам следует: при оптимальном
постоянном шаге $\eta = 1/(\lambda_{\max} + \lambda_{\min})$ ошибка убывает как

$$
\|\theta^{(t)} - \theta^*\| \;\le\; \Bigl(\frac{\kappa - 1}{\kappa + 1}\Bigr)^{t}
\,\|\theta^{(0)} - \theta^*\|,
\qquad \kappa = \mathrm{cond}(X^{\mathsf T}X) = \frac{\lambda_{\max}}{\lambda_{\min}} .
$$

Значит, число итераций до заданной точности растёт **линейно по $\kappa$**.
Проверьте это: постройте матрицы с заданным $\kappa$ и измерьте число итераций
до $\|\theta^{(t)} - \theta^*\| < 10^{-6}\|\theta^*\|$.

In [ ]:
def design_with_cond(n, p, kappa, generator):
    """Матрица X, у которой cond(X^T X) = kappa."""
    # TODO: возьмите случайную матрицу, разложите её по SVD и подмените
    #       сингулярные числа на np.logspace(0, -0.5*log10(kappa), p)
    raise NotImplementedError


def iters_to_converge(X, y, tol=1e-6, max_iter=400_000):
    """Число итераций ГС с шагом 1/(lambda_max + lambda_min) до заданной точности."""
    raise NotImplementedError


# TODO: для kappa из np.logspace(1, 4.5, 8) постройте задачу, измерьте число итераций,
#       выведите таблицу и график в двойном логарифмическом масштабе вместе с
#       эталонной прямой, пропорциональной kappa.

> **Вывод.** Как число итераций зависит от $\kappa$? Во сколько раз медленнее сходится задача с $\kappa = 10^5$ по сравнению с $\kappa = 10$? Что это означает практически для признаков, измеренных в разных единицах?
>
> *(ваш ответ здесь)*

---
# Часть 3. Стохастический градиентный спуск

Определение 2.4 использует **усреднённый** функционал
$Q(\theta) = \frac1\ell\sum_{i=1}^{\ell} Q_i(\theta)$, где
$Q_i(\theta) = (\langle\theta, x_i\rangle - y_i)^2$. Тогда
$\nabla Q_i(\theta) = 2x_i(\langle\theta,x_i\rangle - y_i)$, а для мини-пакета
$B$ градиент усредняется по объектам пакета.

Ключевой факт из лекции: при **постоянном** шаге SGD не сходится, а колеблется
в окрестности $\theta^*$; для сходимости нужен убывающий шаг, удовлетворяющий
условию Роббинса–Монро $\sum\eta_t = \infty$, $\sum\eta_t^2 < \infty$.

Ваш вариант задаёт размеры пакетов `variant["batch_sizes"]` и правило шага
`variant["gd_flavor"]`.

In [ ]:
def sgd(X, y, eta0, n_epochs=40, batch_size=1, schedule="const", generator=None):
    """SGD/mini-batch для УСРЕДНЁННОГО Q. Возвращает (theta, история Q по эпохам).

    schedule: 'const' -- eta_t = eta0; 'sqrt' -- eta0/sqrt(t); 'inverse' -- eta0/t,
    где t -- номер шага (не эпохи).
    """
    raise NotImplementedError


# TODO: 1) на одном графике сравните сходимость при разных batch_size
#          из variant["batch_sizes"] и при полном градиенте (постоянный шаг);
#       2) на втором графике сравните три расписания шага при batch_size=1.
#       По оси Q откладывайте Q - Q*, масштаб логарифмический.
print("правило шага по вашему варианту:", variant["gd_flavor"])

### Задание 3.2. Правило шага из вашего варианта

Реализуйте правило шага, указанное в `variant["gd_flavor"]`:

* **постоянный шаг** — сравните три значения $\eta_0$ и покажите зависимость
  высоты «полки» от $\eta_0$;
* **затухающий шаг $\eta_t = \eta_0/\sqrt t$** — проверьте условие Роббинса–Монро
  для этого расписания (какой из двух рядов сходится, какой расходится?);
* **шаг по Армихо** — реализуйте дробление шага: пока
  $Q(\theta - \eta\nabla Q) > Q(\theta) - c\,\eta\|\nabla Q\|^2$ при $c = 10^{-4}$,
  умножайте $\eta$ на $0.5$; сравните число итераций с постоянным шагом.

In [ ]:
flavor = variant["gd_flavor"]
print("вариант:", flavor)

# TODO: реализуйте задание для СВОЕГО правила шага (см. текст выше).

> **Вывод.** Почему при постоянном шаге SGD не сходится в точку, а мини-пакет размера $\ell$ — сходится? Как высота «полки» зависит от $\eta_0$ и от размера пакета?
>
> *(ваш ответ здесь)*

---
# Часть 4. Мультиколлинеарность

Начнём с примера из лекции 2: $\ell = 3$, $f_1 = (1,2,3)^{\mathsf T}$, $f_2 = 2f_1$.
Тогда

$$
X = \begin{pmatrix}1&2\\2&4\\3&6\end{pmatrix},\qquad
X^{\mathsf T}X = \begin{pmatrix}14&28\\28&56\end{pmatrix},\qquad
\det(X^{\mathsf T}X) = 0 .
$$

Затем перейдём к более реалистичному случаю «почти коллинеарных» признаков
с корреляцией $\rho$ из вашего варианта и проверим утверждение 2.5:

$$
\frac{\|\delta\theta\|}{\|\theta\|} \;\le\; \mathrm{cond}(A)\,
\frac{\|\delta b\|}{\|b\|},\qquad A = X^{\mathsf T}X,\; b = X^{\mathsf T}y .
$$

In [ ]:
# TODO: (1) воспроизведите вырожденный пример лекции: X = [[1,2],[2,4],[3,6]],
#           выведите X^T X, её определитель и ранг, попробуйте np.linalg.solve.
# TODO: (2) постройте почти коллинеарные признаки с rho = variant["collinearity"],
#           решите МНК и выведите cond(X^T X).
# TODO: (3) проверьте неравенство утверждения 2.5: 200 раз возмутите b так, чтобы
#           ||db||/||b|| = 1e-3, и сравните наблюдаемое ||dtheta||/||theta||
#           с оценкой cond(A) * 1e-3.
# TODO: (4) 300 бутстреп-выборок: нарисуйте облако оценок (theta_1, theta_2),
#           выведите их стандартное отклонение и корреляцию.

> **Вывод.** Вдоль какого направления вытянуто облако бутстреп-оценок и почему? Насколько оценка утверждения 2.5 консервативна (во сколько раз медианное возмущение меньше границы)?
>
> *(ваш ответ здесь)*

---
# Часть 5. Гребневая регрессия

Определение 2.7 и теорема 2.9:

$$
Q_\lambda(\theta) = \|X\theta - y\|^2 + \lambda\,\theta^{\mathsf T}D\theta,
\qquad
\theta^*_\lambda = (X^{\mathsf T}X + \lambda D)^{-1}X^{\mathsf T}y,
\qquad
D = \mathrm{diag}(0, 1, \dots, 1).
$$

Матрица $X^{\mathsf T}X + \lambda D$ положительно определена при любом
$\lambda > 0$ — **даже если $X^{\mathsf T}X$ вырождена**. Проверим это на том
самом примере, где `np.linalg.solve` только что отказал.

In [ ]:
def ridge_fit(X, y, lam):
    """theta*_lambda по теореме 2.9. Первый столбец X считается столбцом единиц."""
    # TODO: соберите D = diag(0, 1, ..., 1) и решите (X^T X + lambda D) theta = X^T y
    raise NotImplementedError


# TODO: (1) примените ridge_fit к вырожденной задаче из части 4 при
#           lambda = 0, 1e-8, 1e-3, 1 -- посмотрите, при каком lambda решение появляется
#           и как меняется ||theta||;
#       (2) постройте регуляризационный путь theta_j(lambda) на почти коллинеарных
#           данных для lambdas = eval(variant["lambda_grid"]).

### Задание 5.2. Гребневая регрессия глазами SVD

Пусть данные центрированы (тогда свободный член отделяется и $D = I$),
и $X = U\Sigma V^{\mathsf T}$. Тогда

$$
\theta^*_\lambda \;=\; \sum_{i} \frac{\sigma_i}{\sigma_i^2 + \lambda}\,
(u_i^{\mathsf T} y)\, v_i ,
\qquad\text{при }\lambda = 0:\quad
\theta^* = \sum_i \frac{1}{\sigma_i}(u_i^{\mathsf T}y)\, v_i .
$$

То есть ridge **сжимает** вклад $i$-й компоненты в
$\sigma_i^2/(\sigma_i^2+\lambda)$ раз, и сильнее всего — там, где $\sigma_i$ мало,
то есть по «плохо определённым» направлениям. Проверьте формулу численно и
постройте графики коэффициентов сжатия.

In [ ]:
# TODO: 1) центрируйте X_col и y_col, разложите X по SVD;
#       2) проверьте формулу theta = V diag(s/(s^2+lam)) U^T y для нескольких lambda,
#          сравнив с прямым решением (X^T X + lam I) theta = X^T y;
#       3) постройте графики коэффициентов сжатия s_i^2/(s_i^2 + lambda).

> **Вывод.** По каким направлениям ridge сжимает решение сильнее и почему это разумно? Что происходит с $\|\theta^*_\lambda\|$ при $\lambda \to \infty$?
>
> *(ваш ответ здесь)*

---
# Часть 6. LASSO и покоординатный спуск

Определение 2.12:
$\theta^*_\lambda = \arg\min_\theta\bigl(\|X\theta - y\|^2 + \lambda\sum_{j\ge1}|\theta_j|\bigr)$.
Явной формулы нет, но при фиксированных остальных координатах задача по одной
координате решается точно. Обозначим $r^{(-j)} = y - \sum_{k\ne j} x_k\theta_k$;
тогда минимизируется
$\|r^{(-j)} - x_j\theta_j\|^2 + \lambda|\theta_j|$, и из условия на субградиент

$$
\theta_j \;=\; \frac{S\bigl(\langle x_j, r^{(-j)}\rangle,\; \lambda/2\bigr)}{\|x_j\|^2},
\qquad
S(z, \gamma) = \mathrm{sign}(z)\,\max(|z| - \gamma,\, 0).
$$

Функция $S$ — «мягкий порог»: она **обнуляет** координату, если её вклад меньше
порога. Отсюда и разреженность.

In [ ]:
def soft_threshold(z, gamma):
    """S(z, gamma) = sign(z) * max(|z| - gamma, 0)."""
    raise NotImplementedError


def lasso_cd(X, y, lam, n_iter=300, tol=1e-10):
    """Покоординатный спуск. Первый столбец X -- единицы, он не штрафуется."""
    raise NotImplementedError


from sklearn.linear_model import Lasso

n_obj = 120
X_sp = rng.normal(size=(n_obj, 12))
theta_sparse = np.zeros(12); theta_sparse[[0, 3, 7]] = [3.0, -2.0, 1.5]
y_sp = X_sp @ theta_sparse + rng.normal(0, 0.3, n_obj)
X_sp1 = np.column_stack([np.ones(n_obj), X_sp])

# TODO: сверьте свою реализацию со sklearn.linear_model.Lasso для нескольких lambda.
#       Внимание на нормировку: функционал sklearn -- (1/2n)||y - Xw||^2 + alpha||w||_1,
#       наш -- ||X theta - y||^2 + lambda ||theta||_1. Выразите alpha через lambda.

### Задание 6.2. Ridge против LASSO: пути и разреженность

Постройте на одном рисунке регуляризационные пути обоих методов для разреженной
задачи (истинные ненулевые координаты — три из двенадцати) и график числа
ненулевых координат в зависимости от $\lambda$.

In [ ]:
lambdas = np.logspace(-2, 3.5, 60)

# TODO: постройте пути обоих методов и график числа ненулевых координат от lambda.
#       Выделите цветом те координаты, истинное значение которых не равно нулю
#       (индексы 0, 3, 7 в theta_sparse).

### Задание 6.3. Почему $L_1$ обнуляет, а $L_2$ — нет

Воспроизведите геометрическую картинку из лекции 2: линии уровня $Q(\theta)$
и множества $\{|\theta_1| + |\theta_2| \le c\}$ (ромб) и
$\{\theta_1^2 + \theta_2^2 \le c^2\}$ (круг). Точка касания линии уровня
с ромбом попадает в вершину — а вершины ромба лежат на осях.

In [ ]:
Xg = np.array([[1.0, 0.35], [0.35, 1.0], [0.5, 0.7]])
yg = np.array([1.4, 0.6, 1.0])

# TODO: постройте два рисунка: линии уровня Q и ограничение L1 (ромб) / L2 (круг).
#       Численно найдите минимум Q на границе каждого множества (перебором по границе)
#       и отметьте его. Обратите внимание, где оказывается точка для ромба.

> **Вывод.** Почему точка касания с ромбом почти всегда попадает в вершину, а с кругом — никогда? Как это связано с недифференцируемостью $|\theta_j|$ в нуле?
>
> *(ваш ответ здесь)*

---
# Часть 7. Байесовский взгляд: MAP

Утверждение 2.11: при гауссовском шуме $\varepsilon\sim\mathcal N(0,\sigma^2)$ и
гауссовском априорном распределении $\theta_j \sim \mathcal N(0, \tau^2)$

$$
\theta_{\mathrm{MAP}} = \theta^*_\lambda, \qquad \lambda = \sigma^2/\tau^2 .
$$

Аналогично, лапласовское априорное распределение
$p(\theta_j) \propto e^{-|\theta_j|/b}$ даёт LASSO с $\lambda = 2\sigma^2/b$.

Проверьте обе связи: минимизируйте $-\ln L(\theta) - \ln p(\theta)$ численно
и сравните с формулами.

In [ ]:
sigma, tau, b_lap = 0.3, 0.5, 0.4

# TODO: 1) напишите -ln L(theta) - ln p(theta) для гауссовского и лапласовского
#          априорных распределений (свободный член theta[0] не штрафуется);
#       2) минимизируйте обе функции численно (для лапласовской подойдёт
#          method="Powell" -- функция негладкая);
#       3) сравните с ridge_fit(lambda = sigma^2/tau^2) и lasso_cd(lambda = 2 sigma^2/b);
#       4) нарисуйте обе априорные плотности на одном графике.

> **Вывод.** Совпали ли MAP-оценки с формулами? Посмотрите на графики априорных плотностей: какая из них сильнее «настаивает» на том, что вес равен нулю, и как это объясняет разреженность LASSO?
>
> *(ваш ответ здесь)*

---
# Часть 8. Своя выборка

Примените регуляризацию из вашего варианта (`variant["own_regularizer"]`) к
индивидуальной выборке. Параметр $\lambda$ подбирайте по **отложенной части
обучающей выборки** — контрольную выборку трогать нельзя, она нужна для
финальной оценки. Полноценный скользящий контроль появится в работе 5.

> **Про сетку $\lambda$.** Фиксированная сетка вроде `np.logspace(-3, 4)` —
> частая ошибка: осмысленный масштаб $\lambda$ зависит от масштаба $y$ и $X$.
> Для нашей нормировки штраф начинает действовать, когда $\lambda$ сравним с
> $\langle x_j, y - \overline y\rangle$. Из условия обнуления координаты
> $|\langle x_j, r\rangle| \le \lambda/2$ получается **наименьшее $\lambda$,
> обнуляющее все веса**:
> $$\lambda_{\max} = 2\max_j \bigl|\langle x_j,\, y - \overline y\rangle\bigr| .$$
> Разумная сетка — от $10^{-4}\lambda_{\max}$ до $\lambda_{\max}$: слева МНК,
> справа константа, и весь интересный диапазон гарантированно внутри.

In [ ]:
from sklearn.model_selection import train_test_split

data = load_personal(variant)
Xtr_full, Xte = data["X_train"], data["X_test"]
ytr_full, yte = data["y_train"], data["y_test"]

# TODO: 1) отделите от обучающей выборки валидационную часть (25%);
#       2) постройте сетку от 1e-4 * lambda_max до lambda_max, где
#          lambda_max = 2 * max_j |<x_j, y - mean(y)>| (см. врезку выше),
#          обучите модель из своего варианта и выберите lambda по ошибке
#          на валидационной части;
#       3) переобучите модель с lambda* на ВСЕЙ обучающей выборке;
#       4) сравните на контроле: константа / МНК / ваша регуляризованная модель;
#       5) выведите 8 признаков с наибольшими |коэффициентами| и число ненулевых.

> **Вывод.** Улучшила ли регуляризация качество на контроле по сравнению с чистым МНК? Какое $\lambda^*$ выбрано — ближе к левому или к правому концу сетки?
>
> *(ваш ответ здесь)*

### Задание 8.2. Ситуация, в которой регуляризация решает

Расширьте признаковое пространство всеми попарными произведениями признаков
(`PolynomialFeatures(degree=2)`) и **урежьте обучающую выборку** до 120 объектов.
Теперь $p > \ell$: матрица $X^{\mathsf T}X$ заведомо вырождена, МНК не определён
однозначно, и без регуляризации задача не решается.

Сравните на контроле: МНК (`lstsq`, решение минимальной нормы), Ridge и LASSO
с $\lambda$, подобранным по отложенной части. Здесь допустимо использовать
реализации `sklearn` — свои вы уже проверили в частях 5 и 6.

In [ ]:
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# TODO: 1) расширьте признаки PolynomialFeatures(degree=2, include_bias=False)
#          и отмасштабируйте их StandardScaler; обучающую выборку урежьте до 120 объектов;
#       2) убедитесь, что p > l и ранг X^T X меньше числа признаков;
#       3) подберите alpha для Ridge и Lasso по отложенной части (30 % обучения),
#          сетка от 1e-4 * alpha_max до alpha_max, где
#          alpha_max = max_j |<x_j, y - mean(y)>| / l  (нормировка sklearn);
#       4) сравните на контроле: МНК (lstsq), Ridge, LASSO, константу;
#       5) постройте график MSE(alpha) для обоих методов.

> **Вывод.** Во сколько раз регуляризация выиграла у МНК при $p > \ell$? Сколько признаков оставил LASSO из общего числа? Сравните с заданием 8.1 и сформулируйте правило, когда регуляризация нужна.
>
> *(ваш ответ здесь)*

## Итоги работы

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Утверждение 2.2 требует $\eta < 1/\lambda_{\max}$. Как изменится граница, если минимизировать усреднённый функционал $\frac1\ell\|X\theta-y\|^2$? Проверьте ответ по своему коду.
2. Почему SGD с постоянным шагом не сходится в точку, а мини-пакетный с $|B| = \ell$ — сходится? Что даёт компромисс по размеру пакета?
3. Два признака имеют корреляцию 0.999. Что произойдёт с оценками их коэффициентов и с их суммой $\theta_1 + \theta_2$? Какая из двух величин оценивается устойчиво?
4. Почему в определении 2.7 свободный член не штрафуется? Что произошло бы с решением, если прибавить ко всем $y_i$ константу 1000, а $\theta_0$ при этом штрафовать?
5. Ridge и LASSO дают одинаковое качество на вашей задаче. Какой из них вы выберете и почему? Назовите по одному аргументу за каждый.

### Домашнее задание

1. **Ridge через расширенную выборку.** Докажите, что решение гребневой регрессии совпадает с обычным МНК для выборки, дополненной $n$ «искусственными» объектами: к матрице $X$ снизу приписывается $\sqrt{\lambda}\,\tilde D$ (где $\tilde D$ — единичная матрица без строки свободного члена), а к $y$ — нули. Проверьте численно, что `fit_lstsq` на расширенной матрице даёт то же, что `ridge_fit`. Объясните, почему такой приём делает задачу невырожденной при любом $\lambda>0$.

2. **Elastic Net покоординатным спуском.** Обобщите `lasso_cd` на функционал $\|X\theta-y\|^2 + \lambda_1\sum|\theta_j| + \lambda_2\sum\theta_j^2$: выведите формулу покоординатного обновления (она отличается от LASSO только знаменателем) и сверьтесь со `sklearn.linear_model.ElasticNet`. Постройте пути для трёх значений `l1_ratio` на двух почти дублирующих друг друга признаках и объясните «эффект группировки»: почему LASSO выбирает из пары один признак произвольно, а Elastic Net делит вес между обоими.